In [1]:
"""
Stage 3 (PD vs HC, class-aware) - Feature Selection via Bagged LASSO
=====================================================================

Phase 1 - Pre-fold
    1. Drop strict zero-variance genes (var = 0 across train samples).
    2. Round VST values to 4 decimal places.

Phase 2 - Outer CV (10 outer x 3 seeds = 30 outer folds)
    Outer-fold seeds: 42, 123, 256
    For each outer fold:
        A. Mirror OLS residualization on (sex + cell_type), fit-and-transform.
        B. Variance pre-screen: keep top-V genes by variance on outer-train.
        C. MI ranking: keep top-K of those by mutual_info_classif vs class.
        D. Bagged LASSO stability selection on the top-K:
            - B bootstrap subsamples of 50% size, multinomial L1 over a
              small lambda grid, warm-started from sparser to denser.
            - Per-gene selection frequency = MAX over lambda of fraction of
              bootstraps where any class coef != 0.
            - Selected = freq >= pi_b.

Phase 3 - Consensus panel
    fold_inclusion[g] = (# outer folds in which g was selected) / 30
    Final panel: fold_inclusion >= 0.80
    Stability path saved at thresholds [0.60, 0.70, 0.80, 0.90, 0.95].

Inputs:
    Stage 2 outputs at PDHC new/PDHC-class-aware/Outputs/stage2_split_class_aware_PDHC/
        train.csv      (genes x train samples, raw VST)
        meta_train.csv (sample-aligned metadata: condition, sex, cell_type)

Outputs in: PDHC new/PDHC-class-aware/Outputs/stage3_feature_selection_PDHC/
"""

from __future__ import annotations

import json
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

# ----------------------------- Configuration --------------------------------
PIPELINE_DIR    = Path(r"C:\Users\hafsa\Python PD Project\MI_BaggedLASSO Pipeline\PDHC new")
CLASS_AWARE_DIR = PIPELINE_DIR / "PDHC-class-aware"
SPLIT_DIR       = CLASS_AWARE_DIR / "Outputs" / "stage2_split_class_aware_PDHC"
INPUT_TRAIN     = SPLIT_DIR / "train.csv"
META_TRAIN      = SPLIT_DIR / "meta_train.csv"
OUTPUT_DIR      = CLASS_AWARE_DIR / "Outputs" / "stage3_feature_selection_PDHC"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONDITION_COL   = "condition"
CONFOUNDER_COLS = ["sex", "cell_type"]
SEEDS           = [42, 123, 256]


@dataclass
class Config:
    n_outer:       int          = 10

    # Phase 1
    vst_round:     int          = 4

    # Phase 2B - variance pre-screen before MI
    var_prefilter_k: int        = 5000
    mi_top_k:        int        = 1000

    # Phase 2D - Bagged LASSO stability selection
    B_bootstraps:  int          = 50
    boot_frac:     float        = 0.5
    lambda_grid:   np.ndarray   = field(default_factory=lambda:
                                        np.logspace(-2, 0.3, 3))
    pi_b_default:  float        = 0.6

    # Inner CV (only used when tune_pib=True)
    tune_pib:      bool         = False
    n_inner:       int          = 2
    B_inner:       int          = 30
    pi_b_grid:     List[float]  = field(default_factory=lambda: [0.6, 0.7])

    # LASSO solver knobs
    lasso_max_iter: int         = 1000
    lasso_tol:      float       = 1e-2

    # Phase 3 thresholds for stability path
    consensus_threshold: float  = 0.80
    path_thresholds:     List[float] = field(default_factory=lambda:
                                              [0.60, 0.70, 0.80, 0.90, 0.95])

    # Numerics
    n_jobs:        int          = -1


# ------------------------------ I/O helpers ---------------------------------
def load_inputs() -> Tuple[np.ndarray, np.ndarray, pd.DataFrame, List[str]]:
    df = pd.read_csv(INPUT_TRAIN, index_col=0)
    meta = pd.read_csv(META_TRAIN, index_col=0)
    n_samples_meta = len(meta)
    if df.shape[0] == n_samples_meta and df.shape[1] != n_samples_meta:
        df = df.T

    gene_names = df.index.tolist()
    sample_ids = df.columns.tolist()
    meta = meta.loc[sample_ids]

    X = df.T.values.astype(np.float64)
    y = meta[CONDITION_COL].values
    return X, y, meta, gene_names


# --------------------------- Phase 1 helpers --------------------------------
def drop_zero_variance(X, gene_names):
    var = X.var(axis=0)
    keep_mask = var > 0
    return X[:, keep_mask], keep_mask, [g for g, k in zip(gene_names, keep_mask) if k]


def round_vst(X, decimals=4):
    return np.round(X, decimals=decimals)


# --------------------- Phase 2A: mirror OLS residualization -----------------
def fit_mirror_ols(X_train, meta_train, conf_cols):
    enc = OneHotEncoder(drop="first", sparse_output=False,
                        handle_unknown="ignore")
    enc.fit(meta_train[conf_cols])

    def design(meta_block):
        D = enc.transform(meta_block[conf_cols])
        return np.column_stack([np.ones(len(meta_block)), D])

    C_train = design(meta_train)
    B_ols = np.linalg.pinv(C_train) @ X_train

    def residualize(X, meta_block):
        C = design(meta_block)
        return X - C[:, 1:] @ B_ols[1:, :]

    return residualize, B_ols


# ---------- Phase 2B: variance pre-screen --> Phase 2C: MI top-K -----------
def variance_prescreen(X, k):
    if k >= X.shape[1]:
        return np.ones(X.shape[1], dtype=bool)
    var = X.var(axis=0)
    keep = np.zeros(X.shape[1], dtype=bool)
    keep[np.argsort(var)[-k:]] = True
    return keep


def mi_topk_filter(X, y, k, seed):
    mi = mutual_info_classif(X, y, random_state=seed,
                             discrete_features=False, n_neighbors=3)
    n = X.shape[1]
    keep = np.zeros(n, dtype=bool)
    if k >= n:
        keep[:] = True
    else:
        keep[np.argsort(mi)[-k:]] = True
    return keep, mi


# ------------------- Phase 2D: Bagged LASSO stability selection -------------
def _fit_one_bootstrap(X, y_int, lambda_grid, boot_frac, seed,
                       max_iter, tol):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = rng.choice(n, size=int(np.ceil(boot_frac * n)), replace=False)
    Xb, yb = X[idx], y_int[idx]
    if len(np.unique(yb)) < 2:
        return np.zeros((X.shape[1], len(lambda_grid)), dtype=bool)

    order = np.argsort(lambda_grid)[::-1]
    sel = np.zeros((X.shape[1], len(lambda_grid)), dtype=bool)

    clf = LogisticRegression(
        penalty="l1", solver="saga",
        multi_class="multinomial",
        warm_start=True,
        max_iter=max_iter, tol=tol,
        random_state=seed,
    )
    for li in order:
        lam = lambda_grid[li]
        clf.C = 1.0 / max(lam, 1e-8)
        try:
            clf.fit(Xb, yb)
        except Exception:
            continue
        sel[:, li] = (np.abs(clf.coef_) > 1e-10).any(axis=0)
    return sel


def bagged_lasso_stability(X, y, B, lambda_grid, boot_frac, n_jobs,
                           seed, max_iter, tol):
    classes = np.unique(y)
    cls_to_int = {c: i for i, c in enumerate(classes)}
    y_int = np.array([cls_to_int[c] for c in y])

    seeds = np.random.SeedSequence(seed).generate_state(B)
    results = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(_fit_one_bootstrap)(X, y_int, lambda_grid, boot_frac,
                                    int(s), max_iter, tol)
        for s in seeds
    )
    sel_stack = np.stack(results, axis=0)
    return sel_stack.mean(axis=0).max(axis=1)


def panel_at_threshold(pi_max, gene_names, threshold):
    return [g for g, p in zip(gene_names, pi_max) if p >= threshold]


def jaccard(a, b):
    sa, sb = set(a), set(b)
    return 1.0 if not (sa or sb) else len(sa & sb) / len(sa | sb)


# --------- Inner CV (only invoked when cfg.tune_pib=True) -------------------
def tune_pi_b_inner(X_outer_train, y_outer_train, meta_outer_train,
                    gene_names, cfg, seed):
    skf = StratifiedKFold(n_splits=cfg.n_inner, shuffle=True,
                          random_state=seed)
    inner_panels = {pb: [] for pb in cfg.pi_b_grid}

    for fold, (in_tr, _) in enumerate(skf.split(X_outer_train, y_outer_train)):
        X_in = X_outer_train[in_tr]
        y_in = y_outer_train[in_tr]
        m_in = meta_outer_train.iloc[in_tr]

        residualize_in, _ = fit_mirror_ols(X_in, m_in, CONFOUNDER_COLS)
        X_in_res = residualize_in(X_in, m_in)

        var_keep = variance_prescreen(X_in_res, cfg.var_prefilter_k)
        X_in_var = X_in_res[:, var_keep]
        var_genes = [g for g, k in zip(gene_names, var_keep) if k]

        mi_keep, _ = mi_topk_filter(X_in_var, y_in, cfg.mi_top_k,
                                    seed=seed * 1000 + fold)
        X_in_mi = X_in_var[:, mi_keep]
        mi_genes = [g for g, k in zip(var_genes, mi_keep) if k]
        if X_in_mi.shape[1] == 0:
            for pb in cfg.pi_b_grid:
                inner_panels[pb].append([])
            continue

        pi_max = bagged_lasso_stability(
            X_in_mi, y_in, B=cfg.B_inner,
            lambda_grid=cfg.lambda_grid, boot_frac=cfg.boot_frac,
            n_jobs=cfg.n_jobs, seed=seed * 1000 + fold,
            max_iter=cfg.lasso_max_iter, tol=cfg.lasso_tol)
        for pb in cfg.pi_b_grid:
            inner_panels[pb].append(panel_at_threshold(pi_max, mi_genes, pb))

    score = {}
    for pb, panels in inner_panels.items():
        pairs = [(i, j) for i in range(len(panels)) for j in range(i + 1, len(panels))]
        score[pb] = float(np.mean([jaccard(panels[i], panels[j]) for i, j in pairs])) if pairs else 0.0
    best_pi_b = max(score, key=score.get)
    return best_pi_b, {"jaccard_per_pi_b": score,
                       "panel_size_per_pi_b": {pb: int(np.mean([len(p) for p in panels]))
                                               for pb, panels in inner_panels.items()}}


# ----------------------------- Outer fold core ------------------------------
def run_outer_fold(X, y, meta, gene_names, train_idx, test_idx,
                   fold_id, seed, cfg):
    X_tr = X[train_idx]
    y_tr = y[train_idx]
    m_tr = meta.iloc[train_idx]

    residualize, _ = fit_mirror_ols(X_tr, m_tr, CONFOUNDER_COLS)
    X_tr_res = residualize(X_tr, m_tr)

    var_keep = variance_prescreen(X_tr_res, cfg.var_prefilter_k)
    X_tr_var = X_tr_res[:, var_keep]
    var_genes = [g for g, k in zip(gene_names, var_keep) if k]
    n_var = int(var_keep.sum())

    mi_keep, mi_scores = mi_topk_filter(X_tr_var, y_tr, cfg.mi_top_k, seed=seed)
    X_tr_mi = X_tr_var[:, mi_keep]
    mi_genes = [g for g, k in zip(var_genes, mi_keep) if k]
    n_mi = int(mi_keep.sum())

    if cfg.tune_pib:
        best_pi_b, inner_diag = tune_pi_b_inner(
            X_tr_res, y_tr, m_tr, gene_names, cfg, seed=seed)
    else:
        best_pi_b, inner_diag = cfg.pi_b_default, None

    pi_max = bagged_lasso_stability(
        X_tr_mi, y_tr, B=cfg.B_bootstraps,
        lambda_grid=cfg.lambda_grid, boot_frac=cfg.boot_frac,
        n_jobs=cfg.n_jobs, seed=seed,
        max_iter=cfg.lasso_max_iter, tol=cfg.lasso_tol)
    panel = panel_at_threshold(pi_max, mi_genes, best_pi_b)

    return {
        "fold_id":    fold_id,
        "n_var":      n_var,
        "n_mi":       n_mi,
        "best_pi_b":  best_pi_b,
        "panel":      panel,
        "mi_genes":   mi_genes,
        "pi_max":     pi_max.tolist(),
        "inner_diag": inner_diag,
    }


# ------------------------------ Driver --------------------------------------
def main():
    cfg = Config()
    print("=" * 72)
    print("STAGE 3 (PD vs HC, class-aware) - Feature Selection (Bagged LASSO)")
    print(f"Seeds: {SEEDS}  ({len(SEEDS)} x {cfg.n_outer} = "
          f"{len(SEEDS) * cfg.n_outer} outer folds)")
    print(f"TUNE_PIB={cfg.tune_pib}  pi_b={cfg.pi_b_default if not cfg.tune_pib else cfg.pi_b_grid}")
    print(f"var_prefilter={cfg.var_prefilter_k}, MI top-K={cfg.mi_top_k}")
    print(f"B={cfg.B_bootstraps}, lambdas={len(cfg.lambda_grid)}")
    print("=" * 72)

    X, y, meta, all_gene_names = load_inputs()
    print(f"Loaded train: {X.shape[0]} samples x {X.shape[1]} genes")
    print(f"Class balance: {pd.Series(y).value_counts().to_dict()}")

    # Phase 1
    X_phase1, kept_mask, gene_names = drop_zero_variance(X, all_gene_names)
    n_dropped_zv = int((~kept_mask).sum())
    X_phase1 = round_vst(X_phase1, decimals=cfg.vst_round)
    print(f"\nPhase 1 - zero-variance filter + round to {cfg.vst_round} decimals")
    print(f"  Dropped: {n_dropped_zv}; Kept: {X_phase1.shape[1]}")
    pd.DataFrame({"gene": all_gene_names,
                  "kept_phase1": kept_mask}).to_csv(
        OUTPUT_DIR / "phase1_zero_variance_filter.csv", index=False)

    # Phase 2 outer folds
    print(f"\nPhase 2 - Outer CV folds")
    fold_results = []
    t_start = time.time()
    for seed_i, seed in enumerate(SEEDS):
        skf = StratifiedKFold(n_splits=cfg.n_outer, shuffle=True,
                              random_state=seed)
        for k, (tr_idx, te_idx) in enumerate(skf.split(X_phase1, y)):
            fold_id = f"seed{seed}_fold{k + 1}"
            t0 = time.time()
            print(f"  [{fold_id}] n_train={len(tr_idx)}", end=" ", flush=True)
            res = run_outer_fold(
                X_phase1, y, meta, gene_names,
                train_idx=tr_idx, test_idx=te_idx,
                fold_id=fold_id, seed=seed * 1000 + k, cfg=cfg)
            elapsed = time.time() - t0
            print(f"-> var={res['n_var']}, MI={res['n_mi']}, "
                  f"pi_b={res['best_pi_b']}, panel={len(res['panel'])}  "
                  f"({elapsed:.0f}s)")
            fold_results.append(res)
    total_elapsed = time.time() - t_start
    print(f"  Total Phase 2 time: {total_elapsed/60:.1f} min")

    # Phase 3
    print(f"\nPhase 3 - Consensus panel")
    n_folds = len(fold_results)
    inclusion = pd.Series(0, index=gene_names, dtype=float)
    for r in fold_results:
        for g in r["panel"]:
            inclusion[g] += 1
    inclusion /= n_folds

    path_rows = []
    for thr in cfg.path_thresholds:
        n = int((inclusion >= thr).sum())
        path_rows.append({"threshold": thr, "panel_size": n})
        print(f"  fold_inclusion >= {thr:.2f}  ->  panel size = {n}")
    pd.DataFrame(path_rows).to_csv(OUTPUT_DIR / "stability_path.csv",
                                   index=False)

    consensus = inclusion[inclusion >= cfg.consensus_threshold] \
                    .sort_values(ascending=False)
    print(f"\nFinal consensus panel (>= {cfg.consensus_threshold:.2f}): "
          f"{len(consensus)} genes")

    inclusion.sort_values(ascending=False).to_frame("fold_inclusion") \
             .to_csv(OUTPUT_DIR / "fold_inclusion_per_gene.csv")
    consensus.to_frame("fold_inclusion") \
             .to_csv(OUTPUT_DIR / "consensus_panel.csv")

    pd.DataFrame([{"fold_id": r["fold_id"], "n_var": r["n_var"],
                   "n_mi": r["n_mi"], "best_pi_b": r["best_pi_b"],
                   "panel_size": len(r["panel"])}
                  for r in fold_results]).to_csv(
        OUTPUT_DIR / "fold_summary.csv", index=False)

    with open(OUTPUT_DIR / "fold_panels.json", "w") as f:
        json.dump([{"fold_id":   r["fold_id"],
                    "panel":     r["panel"],
                    "best_pi_b": r["best_pi_b"],
                    "n_mi":      r["n_mi"],
                    "inner_diag": r["inner_diag"]}
                   for r in fold_results], f, indent=2)

    mi_inclusion = pd.Series(0, index=gene_names, dtype=float)
    for r in fold_results:
        for g in r["mi_genes"]:
            mi_inclusion[g] += 1
    mi_inclusion /= n_folds
    mi_inclusion.sort_values(ascending=False).to_frame("mi_inclusion") \
                .to_csv(OUTPUT_DIR / "mi_inclusion_per_gene.csv")

    with open(OUTPUT_DIR / "report.txt", "w") as f:
        f.write("STAGE 3 (PD vs HC, class-aware) - Feature Selection Report\n")
        f.write("=" * 60 + "\n")
        f.write(f"Train samples       : {X.shape[0]}\n")
        f.write(f"Genes (input)       : {X.shape[1]}\n")
        f.write(f"Phase 1 dropped     : {n_dropped_zv} (zero variance)\n")
        f.write(f"Phase 1 kept        : {X_phase1.shape[1]}\n")
        f.write(f"VST rounding        : {cfg.vst_round} decimals\n")
        f.write(f"Outer-fold seeds    : {SEEDS}\n")
        f.write(f"Outer folds         : {n_folds} ({cfg.n_outer} x {len(SEEDS)})\n")
        f.write(f"TUNE_PIB            : {cfg.tune_pib}\n")
        f.write(f"pi_b (default/grid) : {cfg.pi_b_default} / {cfg.pi_b_grid}\n")
        f.write(f"Variance pre-screen : top {cfg.var_prefilter_k}\n")
        f.write(f"MI top-K            : {cfg.mi_top_k}\n")
        f.write(f"B (outer / inner)   : {cfg.B_bootstraps} / {cfg.B_inner}\n")
        f.write(f"lambda grid size    : {len(cfg.lambda_grid)}\n")
        f.write(f"LASSO max_iter / tol: {cfg.lasso_max_iter} / {cfg.lasso_tol}\n")
        f.write(f"Total Phase 2 time  : {total_elapsed/60:.1f} min\n\n")
        f.write("Stability path\n")
        f.write("-" * 30 + "\n")
        for row in path_rows:
            f.write(f"  >= {row['threshold']:.2f}  ->  "
                    f"{row['panel_size']} genes\n")
        f.write(f"\nConsensus threshold : {cfg.consensus_threshold:.2f}\n")
        f.write(f"Consensus panel size: {len(consensus)}\n")

    print(f"\nAll outputs in: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()


STAGE 3 (PD vs HC, class-aware) - Feature Selection (Bagged LASSO)
Seeds: [42, 123, 256]  (3 x 10 = 30 outer folds)
TUNE_PIB=False  pi_b=0.6
var_prefilter=5000, MI top-K=1000
B=50, lambdas=3
Loaded train: 108 samples x 12103 genes
Class balance: {'PD': 72, 'HC': 36}

Phase 1 - zero-variance filter + round to 4 decimals
  Dropped: 0; Kept: 12103

Phase 2 - Outer CV folds
  [seed42_fold1] n_train=97 -> var=5000, MI=1000, pi_b=0.6, panel=1000  (44s)
  [seed42_fold2] n_train=97 -> var=5000, MI=1000, pi_b=0.6, panel=1000  (32s)
  [seed42_fold3] n_train=97 -> var=5000, MI=1000, pi_b=0.6, panel=1000  (34s)
  [seed42_fold4] n_train=97 -> var=5000, MI=1000, pi_b=0.6, panel=1000  (31s)
  [seed42_fold5] n_train=97 -> var=5000, MI=1000, pi_b=0.6, panel=1000  (29s)
  [seed42_fold6] n_train=97 -> var=5000, MI=1000, pi_b=0.6, panel=1000  (30s)
  [seed42_fold7] n_train=97 -> var=5000, MI=1000, pi_b=0.6, panel=1000  (29s)
  [seed42_fold8] n_train=97 -> var=5000, MI=1000, pi_b=0.6, panel=1000  (30s)
  [